# Environment Check

Run this notebook top to bottom **before** starting Module 1. Every cell should finish with a `OK` line. If any cell fails, fix that step before moving on — the later modules all assume this environment works.

**What this notebook verifies:**

1. Python version is 3.10 or newer.
2. All required libraries import and report their versions.
3. Your `.env` file is loaded and `OPENAI_API_KEY` is set.
4. The OpenAI API responds to a minimal request using your key.
5. ChromaDB can create a local collection.

If you installed dependencies via `pip install -r requirements.txt` and copied `.env.example` to `.env` with your key filled in, every cell below should pass.

## 1. Python version

In [ ]:
import sys

required = (3, 10)
current = sys.version_info[:2]
assert current >= required, (
    f"Python {required[0]}.{required[1]}+ required, found {current[0]}.{current[1]}. "
    "Create a fresh virtual environment with a newer Python."
)
print(f"OK: Python {sys.version.split()[0]}")

## 2. Library imports

If any import fails with `ModuleNotFoundError`, re-run `pip install -r requirements.txt` inside your activated virtual environment.

In [ ]:
from importlib.metadata import version

packages = [
    "openai",
    "tiktoken",
    "chromadb",
    "pandas",
    "numpy",
    "python-dotenv",
    "jupyterlab",
]

for name in packages:
    print(f"  {name:<16} {version(name)}")

print("OK: all required libraries installed")

## 3. Load `.env` and check API key

This cell does **not** print your key. It only confirms it is present and looks plausible.

In [ ]:
import os
from dotenv import load_dotenv

loaded = load_dotenv()
api_key = os.getenv("OPENAI_API_KEY", "")

assert loaded, "No .env file found. Copy .env.example to .env and add your key."
assert api_key, "OPENAI_API_KEY is empty. Edit .env and set your key."
assert api_key != "sk-replace-me", "Placeholder key detected. Replace it in .env with your real key."
assert api_key.startswith("sk-"), "OPENAI_API_KEY does not start with 'sk-'. Check the value you pasted."

print(f"OK: OPENAI_API_KEY loaded (length {len(api_key)} chars)")

## 4. Ping the OpenAI API

One tiny chat completion to prove the key works end-to-end. This costs a fraction of a cent.

In [ ]:
from openai import OpenAI

client = OpenAI()
response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    messages=[{"role": "user", "content": "Reply with the single word: pong"}],
    max_tokens=5,
    temperature=0,
)

reply = response.choices[0].message.content.strip()
print(f"Model replied: {reply!r}")
print("OK: OpenAI API reachable with your key")

## 5. ChromaDB

Creates an in-memory collection — no disk writes, no model downloads.

In [ ]:
import chromadb

chroma = chromadb.EphemeralClient()
collection = chroma.get_or_create_collection("env_check")
assert collection.count() == 0
print(f"OK: ChromaDB {chromadb.__version__} works; created collection '{collection.name}'")

## You're ready

If every cell above printed `OK:`, your environment is set up correctly. Open `module_1/01-introduction-to-modern-chatbots.md` to begin.

If you hit a failure:

- **ModuleNotFoundError** — your virtual environment is not activated, or `pip install -r requirements.txt` did not complete. Re-run the install inside the activated venv.
- **No .env file found** — run `cp .env.example .env` and edit the new file to add your real key.
- **`AuthenticationError` from OpenAI** — the key in `.env` is wrong. Regenerate and paste it again, no surrounding quotes or spaces.
- **Any other error** — copy the full traceback and flag it with the course team *before* the first session.